In [ ]:
filename = "Dataset/raw/0077151888593323115-SAME.mp4"

: 

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import urllib.request
import os
import json
from contextlib import ExitStack
from pathlib import Path
from IPython.display import HTML, display

TARGET_FRAME_COUNT = 10
LANDMARK_CHANNELS = ("x", "y", "z", "visibility", "presence")
EXPECTED_HAND_POINTS = 21
EXPECTED_BODY_POINTS = 33
EXPECTED_FACE_POINTS = 478
MISSING_POINT = {
    "x": np.nan,
    "y": np.nan,
    "z": np.nan,
    "visibility": 0.0,
    "presence": 0.0,
}

# --- Model paths and download ---
HAND_MODEL_PATH = "hand_landmarker.task"
POSE_MODEL_PATH = "pose_landmarker.task"
FACE_MODEL_PATH = "face_landmarker.task"

if not os.path.exists(HAND_MODEL_PATH):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        HAND_MODEL_PATH,
    )
if not os.path.exists(POSE_MODEL_PATH):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task",
        POSE_MODEL_PATH,
    )
if not os.path.exists(FACE_MODEL_PATH):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        FACE_MODEL_PATH,
    )

# --- MediaPipe task options ---
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode


def create_base_options(model_path, delegate):
    return BaseOptions(model_asset_path=model_path, delegate=delegate)


def build_hand_options(delegate):
    return HandLandmarkerOptions(
        base_options=create_base_options(HAND_MODEL_PATH, delegate),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )


def build_pose_options(delegate):
    return PoseLandmarkerOptions(
        base_options=create_base_options(POSE_MODEL_PATH, delegate),
        running_mode=VisionRunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )


def build_face_options(delegate):
    return FaceLandmarkerOptions(
        base_options=create_base_options(FACE_MODEL_PATH, delegate),
        running_mode=VisionRunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )


def create_landmarker(factory, options_builder, name):
    delegates = [BaseOptions.Delegate.GPU, BaseOptions.Delegate.CPU]
    errors = []
    for delegate in delegates:
        try:
            return factory.create_from_options(options_builder(delegate)), delegate
        except (OSError, RuntimeError) as exc:
            errors.append(f"{delegate.name}: {exc}")
    raise RuntimeError(f"Could not create {name}. Attempts: {' | '.join(errors)}")


def frame_to_mp_image(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)


def load_video_frames_numpy(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video file: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()

    if not frames:
        raise ValueError(f"No frames were read from {video_path}")
    return np.stack(frames, axis=0), float(fps)


def sample_frame_indices(total_frames, sample_count=TARGET_FRAME_COUNT):
    if total_frames <= 0 or sample_count <= 0:
        return []
    indices = np.rint(np.linspace(0, total_frames - 1, sample_count)).astype(np.int64)
    indices[0] = 0
    indices[-1] = total_frames - 1
    return indices.tolist()


def make_timestamps_ms(sampled_indices, fps):
    frame_ms = max(1, int(round(1000.0 / fps)))
    timestamps = []
    previous = -1
    for source_frame_index in sampled_indices:
        timestamp = int(round((source_frame_index / fps) * 1000.0))
        if timestamp <= previous:
            timestamp = previous + frame_ms
        timestamps.append(timestamp)
        previous = timestamp
    return timestamps


def serialize_landmark(landmark):
    visibility = getattr(landmark, "visibility", 0.0) or 0.0
    presence = getattr(landmark, "presence", 0.0) or 0.0
    return {
        "x": float(landmark.x),
        "y": float(landmark.y),
        "z": float(landmark.z),
        "visibility": float(visibility),
        "presence": float(presence),
    }


def serialize_landmark_list(landmark_list):
    return [serialize_landmark(landmark) for landmark in landmark_list]


def serialize_categories(category_list):
    return [
        {
            "index": category.index,
            "score": float(category.score),
            "category_name": category.category_name,
            "display_name": category.display_name,
        }
        for category in category_list
    ]


def serialize_hand_result(hand_result):
    hand_landmarks = getattr(hand_result, "hand_landmarks", []) or []
    handedness = getattr(hand_result, "handedness", []) or []
    return [
        {
            "landmarks": serialize_landmark_list(landmarks),
            "handedness": serialize_categories(handedness[index]) if index < len(handedness) else [],
        }
        for index, landmarks in enumerate(hand_landmarks)
    ]


def serialize_pose_result(pose_result):
    pose_landmarks = getattr(pose_result, "pose_landmarks", []) or []
    return [serialize_landmark_list(landmarks) for landmarks in pose_landmarks]


def serialize_face_result(face_result):
    face_landmarks = getattr(face_result, "face_landmarks", []) or []
    return [serialize_landmark_list(landmarks) for landmarks in face_landmarks]


def classify_hands(hand_entries):
    hands = {"left": [], "right": []}
    for entry in hand_entries:
        handedness = entry.get("handedness", [])
        label = "unknown"
        if handedness:
            category_name = (handedness[0].get("category_name") or "").lower()
            if category_name in hands:
                label = category_name

        landmarks = entry.get("landmarks", [])
        if label in hands and not hands[label]:
            hands[label] = landmarks
        elif not hands["left"]:
            hands["left"] = landmarks
        elif not hands["right"]:
            hands["right"] = landmarks
    return hands


def normalize_landmarks(landmarks, expected_len, fallback=None):
    points = list(landmarks or [])
    if not points and fallback is not None:
        points = [dict(point) for point in fallback]

    if len(points) < expected_len:
        points.extend(dict(MISSING_POINT) for _ in range(expected_len - len(points)))
    elif len(points) > expected_len:
        points = points[:expected_len]

    return [
        {
            "x": float(point.get("x", np.nan)),
            "y": float(point.get("y", np.nan)),
            "z": float(point.get("z", np.nan)),
            "visibility": float(point.get("visibility", 0.0)),
            "presence": float(point.get("presence", 0.0)),
        }
        for point in points
    ]


def has_valid_x_coordinate(landmarks):
    return not np.isnan([point["x"] for point in landmarks]).all()


def landmarks_to_array(landmarks):
    return np.array(
        [[point[channel] for channel in LANDMARK_CHANNELS] for point in landmarks],
        dtype=np.float32,
    )


def draw_hand(bgr_image, hand_result, in_place=False):
    annotated = bgr_image if in_place else np.copy(bgr_image)
    if hasattr(hand_result, "hand_landmarks") and hand_result.hand_landmarks:
        hand_style = drawing_styles.get_default_hand_landmarks_style()
        hand_conn_style = drawing_styles.get_default_hand_connections_style()
        for landmarks in hand_result.hand_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_style,
                connection_drawing_spec=hand_conn_style,
            )
    return annotated


def draw_pose(bgr_image, pose_result, in_place=False):
    annotated = bgr_image if in_place else np.copy(bgr_image)
    if hasattr(pose_result, "pose_landmarks") and pose_result.pose_landmarks:
        pose_style = drawing_styles.get_default_pose_landmarks_style()
        pose_conn_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)
        for landmarks in pose_result.pose_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
                landmark_drawing_spec=pose_style,
                connection_drawing_spec=pose_conn_style,
            )
    return annotated


def draw_face(bgr_image, face_result, in_place=False):
    annotated = bgr_image if in_place else np.copy(bgr_image)
    if hasattr(face_result, "face_landmarks") and face_result.face_landmarks:
        tesselation_style = drawing_styles.get_default_face_mesh_tesselation_style()
        contours_style = drawing_styles.get_default_face_mesh_contours_style()
        iris_style = drawing_styles.get_default_face_mesh_iris_connections_style()
        for landmarks in face_result.face_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=tesselation_style,
            )
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=contours_style,
            )
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
                landmark_drawing_spec=None,
                connection_drawing_spec=iris_style,
            )
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
                landmark_drawing_spec=None,
                connection_drawing_spec=iris_style,
            )
    return annotated


# 1. Load all frames to NumPy.
file_path = filename
video_stem = Path(file_path).stem
all_source_frames_bgr, fps = load_video_frames_numpy(file_path)
source_frame_count = len(all_source_frames_bgr)
print(f"1/4 Loaded all frames to NumPy: {all_source_frames_bgr.shape} BGR frames at {fps:.2f} fps.")

# 2. Sample the loaded NumPy frames into exactly 32 frames.
sampled_source_frame_indices = sample_frame_indices(source_frame_count, TARGET_FRAME_COUNT)
sampled_frames_bgr = all_source_frames_bgr[sampled_source_frame_indices].copy()
sampled_timestamps_ms = make_timestamps_ms(sampled_source_frame_indices, fps)
print(
    f"2/4 Sampled {source_frame_count} frames into {len(sampled_frames_bgr)} frames. "
    f"First source frame={sampled_source_frame_indices[0]}, "
    f"last source frame={sampled_source_frame_indices[-1]}."
)

# 3. Generate hand, body, and face skeletons from the 32 sampled frames.
preview_original_frames = []
preview_hand_frames = []
preview_body_frames = []
preview_face_frames = []
preview_all_frames = []
keypoints_by_frame = []
hand_skeleton_frames = []
body_skeleton_frames = []
face_skeleton_frames = []
last_valid = {
    "left_hand": None,
    "right_hand": None,
    "body": None,
    "face": None,
}

with ExitStack() as stack:
    hand_landmarker, hand_delegate = create_landmarker(HandLandmarker, build_hand_options, "hand landmarker")
    pose_landmarker, pose_delegate = create_landmarker(PoseLandmarker, build_pose_options, "pose landmarker")
    face_landmarker, face_delegate = create_landmarker(FaceLandmarker, build_face_options, "face landmarker")

    hand_landmarker = stack.enter_context(hand_landmarker)
    pose_landmarker = stack.enter_context(pose_landmarker)
    face_landmarker = stack.enter_context(face_landmarker)

    print(f"Delegates used -> hand: {hand_delegate.name}, body: {pose_delegate.name}, face: {face_delegate.name}")

    for sample_position, (source_frame_index, timestamp_ms, frame_bgr) in enumerate(
        zip(sampled_source_frame_indices, sampled_timestamps_ms, sampled_frames_bgr)
    ):
        mp_image = frame_to_mp_image(frame_bgr)
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
        face_result = face_landmarker.detect_for_video(mp_image, timestamp_ms)

        serialized_hands = serialize_hand_result(hand_result)
        serialized_body = serialize_pose_result(pose_result)
        serialized_face = serialize_face_result(face_result)

        hands = classify_hands(serialized_hands)
        left_hand = normalize_landmarks(hands.get("left", []), EXPECTED_HAND_POINTS, fallback=last_valid["left_hand"])
        right_hand = normalize_landmarks(hands.get("right", []), EXPECTED_HAND_POINTS, fallback=last_valid["right_hand"])
        body = normalize_landmarks(serialized_body[0] if serialized_body else [], EXPECTED_BODY_POINTS, fallback=last_valid["body"])
        face = normalize_landmarks(serialized_face[0] if serialized_face else [], EXPECTED_FACE_POINTS, fallback=last_valid["face"])

        if has_valid_x_coordinate(left_hand):
            last_valid["left_hand"] = [dict(point) for point in left_hand]
        if has_valid_x_coordinate(right_hand):
            last_valid["right_hand"] = [dict(point) for point in right_hand]
        if has_valid_x_coordinate(body):
            last_valid["body"] = [dict(point) for point in body]
        if has_valid_x_coordinate(face):
            last_valid["face"] = [dict(point) for point in face]

        hand_skeleton_frames.append(
            np.stack([landmarks_to_array(left_hand), landmarks_to_array(right_hand)], axis=0)
        )
        body_skeleton_frames.append(landmarks_to_array(body))
        face_skeleton_frames.append(landmarks_to_array(face))

        preview_original_frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        preview_hand_frames.append(cv2.cvtColor(draw_hand(frame_bgr, hand_result), cv2.COLOR_BGR2RGB))

        body_preview = draw_hand(frame_bgr, hand_result)
        body_preview = draw_pose(body_preview, pose_result, in_place=True)
        preview_body_frames.append(cv2.cvtColor(body_preview, cv2.COLOR_BGR2RGB))

        face_preview = draw_hand(frame_bgr, hand_result)
        face_preview = draw_face(face_preview, face_result, in_place=True)
        preview_face_frames.append(cv2.cvtColor(face_preview, cv2.COLOR_BGR2RGB))

        all_preview = draw_hand(frame_bgr, hand_result)
        all_preview = draw_pose(all_preview, pose_result, in_place=True)
        all_preview = draw_face(all_preview, face_result, in_place=True)
        preview_all_frames.append(cv2.cvtColor(all_preview, cv2.COLOR_BGR2RGB))

        keypoints_by_frame.append(
            {
                "sample_position": sample_position,
                "frame_index": int(source_frame_index),
                "timestamp_ms": int(timestamp_ms),
                "hand_landmarks": serialized_hands,
                "pose_landmarks": serialized_body,
                "face_landmarks": serialized_face,
            }
        )

hand_skeleton = np.stack(hand_skeleton_frames, axis=0)
body_skeleton = np.stack(body_skeleton_frames, axis=0)
face_skeleton = np.stack(face_skeleton_frames, axis=0)
preview_original_frames = np.stack(preview_original_frames, axis=0)
preview_hand_frames = np.stack(preview_hand_frames, axis=0)
preview_body_frames = np.stack(preview_body_frames, axis=0)
preview_face_frames = np.stack(preview_face_frames, axis=0)
preview_all_frames = np.stack(preview_all_frames, axis=0)

keypoints_output_path = f"{video_stem}.json"
hand_skeleton_path = f"{video_stem}_hand_32.npy"
body_skeleton_path = f"{video_stem}_body_32.npy"
face_skeleton_path = f"{video_stem}_face_32.npy"

np.save(hand_skeleton_path, hand_skeleton)
np.save(body_skeleton_path, body_skeleton)
np.save(face_skeleton_path, face_skeleton)
with open(keypoints_output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "video_path": file_path,
            "fps": fps,
            "source_frame_count": int(source_frame_count),
            "target_frame_count": TARGET_FRAME_COUNT,
            "num_frames": len(keypoints_by_frame),
            "sampled_source_frame_indices": [int(index) for index in sampled_source_frame_indices],
            "skeleton_shapes": {
                "hand": list(hand_skeleton.shape),
                "body": list(body_skeleton.shape),
                "face": list(face_skeleton.shape),
            },
            "channels": list(LANDMARK_CHANNELS),
            "frames": keypoints_by_frame,
        },
        f,
        indent=2,
    )

print(
    "3/4 Generated 32-frame skeletons: "
    f"hand {hand_skeleton.shape}, body {body_skeleton.shape}, face {face_skeleton.shape}."
)
print(f"Saved: {keypoints_output_path}, {hand_skeleton_path}, {body_skeleton_path}, {face_skeleton_path}")

# 4. Preview the sampled frames with generated skeleton overlays.
num_frames = len(preview_original_frames)
fig, axs = plt.subplots(1, 5, figsize=(20, 4))
titles = ["Original", "Hand", "Hand + Body", "Hand + Face", "Hand + Body + Face"]
preview_sets = [
    preview_original_frames,
    preview_hand_frames,
    preview_body_frames,
    preview_face_frames,
    preview_all_frames,
]
ims = [ax.imshow(frames[0]) for ax, frames in zip(axs, preview_sets)]

for ax, title in zip(axs, titles):
    ax.set_title(title)
    ax.axis("off")


def update_preview(frame_index):
    for image_artist, frames in zip(ims, preview_sets):
        image_artist.set_array(frames[frame_index])
    fig.suptitle(
        f"32-frame skeleton preview | sample {frame_index + 1}/{num_frames} | "
        f"source frame {sampled_source_frame_indices[frame_index]}"
    )
    return ims


ani = animation.FuncAnimation(
    fig,
    update_preview,
    frames=num_frames,
    interval=max(1, int(round(1000 / min(fps, 30)))),
    blit=True,
    repeat=True,
)

plt.tight_layout()
plt.close(fig)
print("4/4 Previewing the 32 generated skeleton frames.")
display(HTML(ani.to_jshtml()))


I0000 00:00:1777402455.595459     791 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777402455.673145     791 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777402456.877826     791 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
# 3D preview from the 32-frame keypoints JSON generated above.
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
from pathlib import Path

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
]

POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10), (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (15, 17), (15, 19), (15, 21),
    (16, 18), (16, 20), (16, 22),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (29, 31),
    (24, 26), (26, 28), (28, 30), (30, 32),
]

EXPECTED_HAND_POINTS = 21
EXPECTED_POSE_POINTS = 33
EXPECTED_FACE_POINTS = 478
MISSING_POINT = {"x": np.nan, "y": np.nan, "z": np.nan, "visibility": 0.0, "presence": 0.0}
TARGET_FRAME_COUNT = globals().get("TARGET_FRAME_COUNT", 32)
SAVE_3D_GIF = True
VIEW_SPECS = [
    ("X+", 0, 0),
    ("X-", 0, 180),
    ("Y+", 0, 90),
    ("Y-", 0, -90),
    ("Z+", 90, -90),
    ("Z-", -90, -90),
]

keypoints_path = Path(globals().get("keypoints_output_path", f"{Path(filename).stem}.json"))
with open(keypoints_path, "r", encoding="utf-8") as f:
    keypoints_data = json.load(f)

frames_data = keypoints_data.get("frames", [])
if not frames_data:
    raise ValueError(f"No frames found in {keypoints_path}")
if len(frames_data) != TARGET_FRAME_COUNT:
    raise ValueError(f"Expected {TARGET_FRAME_COUNT} frames in {keypoints_path}, got {len(frames_data)}")

print(f"3D preview will render {len(frames_data)} skeleton frames from {keypoints_path}.")


def normalize_landmarks(landmarks, expected_len, fallback=None):
    points = list(landmarks or [])
    if not points and fallback is not None:
        points = [dict(point) for point in fallback]
    if len(points) < expected_len:
        points = points + [dict(MISSING_POINT) for _ in range(expected_len - len(points))]
    elif len(points) > expected_len:
        points = points[:expected_len]
    return [
        {
            "x": float(point.get("x", np.nan)),
            "y": float(point.get("y", np.nan)),
            "z": float(point.get("z", np.nan)),
            "visibility": float(point.get("visibility", 0.0)),
            "presence": float(point.get("presence", 0.0)),
        }
        for point in points
    ]


def extract_xyz(landmarks):
    xs = np.array([float(point["x"]) for point in landmarks], dtype=float)
    ys = -np.array([float(point["y"]) for point in landmarks], dtype=float)
    zs = -np.array([float(point["z"]) for point in landmarks], dtype=float)
    return xs, ys, zs


def build_connection_coords(landmarks, connections):
    x_lines, y_lines, z_lines = [], [], []
    for start_idx, end_idx in connections:
        if start_idx >= len(landmarks) or end_idx >= len(landmarks):
            continue
        start = landmarks[start_idx]
        end = landmarks[end_idx]
        if any(np.isnan([start["x"], start["y"], start["z"], end["x"], end["y"], end["z"]])):
            x_lines.extend([np.nan, np.nan, np.nan])
            y_lines.extend([np.nan, np.nan, np.nan])
            z_lines.extend([np.nan, np.nan, np.nan])
            continue
        x_lines.extend([float(start["x"]), float(end["x"]), np.nan])
        y_lines.extend([-float(start["y"]), -float(end["y"]), np.nan])
        z_lines.extend([-float(start["z"]), -float(end["z"]), np.nan])
    return x_lines, y_lines, z_lines


def classify_hands(hand_entries):
    hands = {"left": [], "right": []}
    for entry in hand_entries:
        handedness = entry.get("handedness", [])
        label = "unknown"
        if handedness:
            category_name = (handedness[0].get("category_name") or "").lower()
            if category_name in hands:
                label = category_name
        landmarks = entry.get("landmarks", [])
        if label in hands and not hands[label]:
            hands[label] = landmarks
        elif not hands["left"]:
            hands["left"] = landmarks
        elif not hands["right"]:
            hands["right"] = landmarks
    return hands


def preprocess_frames(frames_data):
    processed = []
    last_valid = {"left_hand": None, "right_hand": None, "pose": None, "face": None}
    for frame_data in frames_data:
        hands = classify_hands(frame_data.get("hand_landmarks", []))
        pose_landmarks = frame_data.get("pose_landmarks", [])
        face_landmarks = frame_data.get("face_landmarks", [])

        left_hand = normalize_landmarks(hands.get("left", []), EXPECTED_HAND_POINTS, fallback=last_valid["left_hand"])
        right_hand = normalize_landmarks(hands.get("right", []), EXPECTED_HAND_POINTS, fallback=last_valid["right_hand"])
        pose = normalize_landmarks(pose_landmarks[0] if pose_landmarks else [], EXPECTED_POSE_POINTS, fallback=last_valid["pose"])
        face = normalize_landmarks(face_landmarks[0] if face_landmarks else [], EXPECTED_FACE_POINTS, fallback=last_valid["face"])

        if not np.isnan([point["x"] for point in left_hand]).all():
            last_valid["left_hand"] = [dict(point) for point in left_hand]
        if not np.isnan([point["x"] for point in right_hand]).all():
            last_valid["right_hand"] = [dict(point) for point in right_hand]
        if not np.isnan([point["x"] for point in pose]).all():
            last_valid["pose"] = [dict(point) for point in pose]
        if not np.isnan([point["x"] for point in face]).all():
            last_valid["face"] = [dict(point) for point in face]

        processed.append(
            {
                "frame_index": frame_data.get("frame_index", len(processed)),
                "hands": {"left": left_hand, "right": right_hand},
                "pose": pose,
                "face": face,
            }
        )
    return processed


def gather_bounds(processed_frames):
    xs_all, ys_all, zs_all = [], [], []
    for frame_data in processed_frames:
        for landmarks in [frame_data["hands"]["left"], frame_data["hands"]["right"], frame_data["pose"], frame_data["face"]]:
            xs, ys, zs = extract_xyz(landmarks)
            valid = ~(np.isnan(xs) | np.isnan(ys) | np.isnan(zs))
            if np.any(valid):
                xs_all.extend(xs[valid].tolist())
                ys_all.extend(ys[valid].tolist())
                zs_all.extend(zs[valid].tolist())
    if not xs_all:
        return (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)
    x_min, x_max = min(xs_all), max(xs_all)
    y_min, y_max = min(ys_all), max(ys_all)
    z_min, z_max = min(zs_all), max(zs_all)
    span = max(x_max - x_min, y_max - y_min, z_max - z_min, 1e-6)
    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2
    half = span / 2
    pad = span * 0.05
    return (
        (x_mid - half - pad, x_mid + half + pad),
        (y_mid - half - pad, y_mid + half + pad),
        (z_mid - half - pad, z_mid + half + pad),
    )


def style_axis(ax, title, bounds, elev, azim):
    x_limits, y_limits, z_limits = bounds
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_zlim(*z_limits)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    try:
        ax.set_box_aspect((1, 1, 1))
    except AttributeError:
        pass


def draw_landmark_group(ax, landmarks, color, size, connections=None, line_width=1.5):
    xs, ys, zs = extract_xyz(landmarks)
    valid = ~(np.isnan(xs) | np.isnan(ys) | np.isnan(zs))
    if np.any(valid):
        ax.scatter(xs[valid], ys[valid], zs[valid], c=color, s=size, depthshade=False)
    if connections:
        x_lines, y_lines, z_lines = build_connection_coords(landmarks, connections)
        ax.plot(x_lines, y_lines, z_lines, color=color, linewidth=line_width, alpha=0.9)


def draw_frame(ax, frame_data):
    draw_landmark_group(ax, frame_data["hands"]["left"], "#d62728", 12, HAND_CONNECTIONS, 1.8)
    draw_landmark_group(ax, frame_data["hands"]["right"], "#2ca02c", 12, HAND_CONNECTIONS, 1.8)
    draw_landmark_group(ax, frame_data["pose"], "#1f77b4", 10, POSE_CONNECTIONS, 1.8)
    draw_landmark_group(ax, frame_data["face"], "#9467bd", 3)


processed_frames = preprocess_frames(frames_data)
bounds = gather_bounds(processed_frames)
fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={"projection": "3d"})
axes = axes.ravel()


def update_3d(frame_index):
    frame_data = processed_frames[frame_index]
    for ax, (title, elev, azim) in zip(axes, VIEW_SPECS):
        ax.cla()
        style_axis(ax, title, bounds, elev, azim)
        draw_frame(ax, frame_data)
    fig.suptitle(f"Hand + Body + Face | source frame {frame_data['frame_index']}")
    return []


update_3d(0)
ani_3d = animation.FuncAnimation(
    fig,
    update_3d,
    frames=len(processed_frames),
    interval=50,
    blit=False,
    repeat=True,
)

plt.tight_layout()
plt.close(fig)
display(HTML(ani_3d.to_jshtml()))

if SAVE_3D_GIF:
    from matplotlib.animation import PillowWriter
    gif_path = f"{Path(filename).stem}.gif"
    ani_3d.save(gif_path, writer=PillowWriter(fps=10))
    print(f"Saved {gif_path}")


3D preview will render 32 skeleton frames from stack4.json.


Saved stack4.gif
